In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sklearn.linear_model as lm
import re
import statsmodels.api as sm
import pandas as pd

import h5py

In [ ]:
data = h5py.File('Randomstim_mouse_forAustinFeb2023.mat','r')

In [ ]:
myDict = {}
for key in data.keys():
    myDict[key] = data[key].value


In [ ]:
alltime_list = []
laser_list = []
lasertime_list = []
norm_list = []
t_list = []
t_norm_list = []
tuse_list = []
usef_list = []
uselaser_list = []
usetimes_list = []

for key in myDict.keys():
    if bool(re.search('_alltime',key)):
        print('Alltime',key)
        alltime_list.append(np.squeeze(myDict[key]))  
    elif bool(re.search('_lasertime',key)):
        print('Lasertime',key)
        lasertime_list.append(np.squeeze(myDict[key])) 
    elif bool(re.search('_laser',key)):
        print('Laser',key)
        laser_list.append(np.squeeze(myDict[key])) 
    elif bool(re.search('_t_norm',key)):print(myDict['Mouse9332_030421_tuse'].shape)
        print('T_norm',key)
        t_norm_list.append(np.squeeze(myDict[key])) 
    elif bool(re.search('_norm',key)):
        print('Norm',key)
        norm_list.append(np.squeeze(myDict[key]))  
    elif bool(re.search('_tuse',key)):
        print('Tuse',key)
        tuse_list.append(np.squeeze(myDict[key])) 
    elif bool(re.search('_t',key)):
        print('T',key)
        t_list.append(np.squeeze(myDict[key]))  
    elif bool(re.search('_usef',key)):
        print('usef',key)
        usef_list.append(np.squeeze(myDict[key].T))     
    elif bool(re.search('_uselaser',key)):
        print('uselaser',key)
        uselaser_list.append(np.squeeze(myDict[key]))
    elif bool(re.search('_usetimes',key)):
        print('usetimes',key)
        usetimes_list.append(np.squeeze(myDict[key]))   
    else:
        print('!!!!!!!!!!!!!!!!!!!!!!!!')
        print('Unmatched',key)

In [ ]:
uselaser = np.concatenate(uselaser_list)
usef = np.vstack(usef_list)
tuse = np.concatenate(tuse_list)
condition = usef[:,1]
behavior = usef[:,2]

idx_select = ((uselaser==1)|(uselaser==0))&(condition==4) # Select blue light and male condition
behavior_select = (behavior==1)|(behavior==2)
idx_total = idx_select & behavior_select
network_sub = tuse[idx_total]
stim_sub = uselaser[idx_total]
behavior_sub = behavior[idx_total]
behavior_sub[behavior_sub==2] = 0


In [ ]:
X1 = pd.DataFrame(data={'Stimulation':stim_sub})
X2 = pd.DataFrame(data={'Stimulation':stim_sub,'Network':network_sub})
Y = pd.DataFrame(data={'Behavior':behavior_sub})

## Predict behavior using stimulation

In [ ]:
X1_2 = sm.add_constant(X1)
reduced_model = sm.Logit(Y,X1_2).fit()

### Log likelihood

In [ ]:
reduced_ll = reduced_model.llf
print('Log likelihood',reduced_ll)

### Coefficients

In [ ]:
reduced_model.summary()

## Predict behavior using stimulation + network score

In [ ]:
X2_2 = sm.add_constant(X2)
full_model = sm.Logit(Y,X1_2).fit()
full_ll = full_model.llf


### Log likelihood

In [ ]:
print(full_ll)

### Coefficients

In [ ]:
full_model.summary()

## Now significance test

In [ ]:
LR_statistic = -2*(reduced_ll-full_ll)
print('Likelihood_ratio stat',LR_statistic)
from scipy.stats import chi2

p_val = chi2.sf(LR_statistic,1)
print('P-value',p_val)

## For completeness, only network

In [ ]:
X3 = pd.DataFrame(data={'Network':network_sub})
X3_2 = sm.add_constant(X3)
network_model = sm.Logit(Y,X3_2).fit()
network_ll = network_model.llf

### Log likelihood

In [ ]:
network_ll

### Coefficients

In [ ]:
network_model.summary()